# Train the CamDetec pest model (free)

Trains a **YOLOv8** pest detector and exports it to **TensorFlow.js**, producing a
`model.json` that drops straight into CamDetec's pest-model slot. Runs free on a
**Colab T4 GPU** (Runtime → Change runtime type → T4 GPU) or a **Kaggle** GPU
notebook. No cost to train, no cost to run — the model runs in the browser.

**The only manual step:** a free Roboflow API key to pull a labeled pest dataset
(cell 3). Everything else is Run → Run all.

Pipeline: install → dataset → train → export to TF.js → download → install in the app.

In [ ]:
%pip install -q ultralytics roboflow pyyaml

## 1. Confirm the GPU

If this prints `cuda … True`, you have a free GPU. On Colab enable it via
Runtime → Change runtime type → T4 GPU. On Kaggle: Settings → Accelerator → GPU.

In [ ]:
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (training will be slow — switch to a GPU runtime)")

## 2. Get a labeled pest dataset

1. Go to **https://universe.roboflow.com** and search e.g. `rat`, `mouse`, `rodent`,
   `cockroach`, `raccoon`. Pick a detection dataset you like (many are public).
2. On that dataset: **Download Dataset → Format: YOLOv8 → "show download code"**.
3. Roboflow shows a 3-line snippet with **your free API key** already filled in.
   Paste it into the next cell (replace the example).

Free Roboflow account: https://app.roboflow.com (30 seconds, no card).

*Tip:* start with one or two classes (rat, mouse) and ≥300 labeled instances each;
add cockroach / raccoon later. Include night/IR frames if your camera runs at night.

Already have a YOLO dataset (images + labels + `data.yaml`)? Skip Roboflow and set
`DATA_YAML` to its `data.yaml` path in the next cell.

In [ ]:
from roboflow import Roboflow

# --- paste YOUR Roboflow snippet here (replace this example) ---
rf = Roboflow(api_key="PASTE_YOUR_FREE_ROBOFLOW_API_KEY")
project = rf.workspace("WORKSPACE_SLUG").project("PROJECT_SLUG")
dataset = project.version(1).download("yolov8")
DATA_YAML = dataset.location + "/data.yaml"
# ---------------------------------------------------------------

# Or, if you already have a YOLO dataset:
# DATA_YAML = "/path/to/your/data.yaml"

print("data.yaml:", DATA_YAML)

## 3. Train

`yolov8n` (nano) is deliberately small so it runs smoothly in the browser. 80
epochs on a T4 takes roughly 20–60 min for a few thousand images; bump epochs if
mAP is still climbing.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")   # start from COCO-pretrained nano weights (transfer learning)
model.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=640,          # MUST stay 640 — the app's YoloTfjsDetector expects imgsz=640
    batch=16,
    patience=20,        # early-stop if it plateaus
    name="camdetec_pest",
)

## 4. Quick check

mAP50 above ~0.5 is a usable start; higher is better. Low numbers usually mean
more/cleaner data is needed, not more epochs.

In [ ]:
metrics = model.val()
print("mAP50:", round(float(metrics.box.map50), 3), "| mAP50-95:", round(float(metrics.box.map), 3))

## 5. Export to TensorFlow.js

Default export (no embedded NMS) at imgsz 640 — the app decodes the raw YOLOv8
head and runs NMS in-browser, so keep it default.

In [ ]:
export_path = model.export(format="tfjs", imgsz=640)
print("TF.js model at:", export_path)

## 6. Write labels.json (class order must match the model)

In [ ]:
import yaml, json, os, glob

with open(DATA_YAML) as f:
    dy = yaml.safe_load(f)
names = dy["names"]
if isinstance(names, dict):
    names = [names[i] for i in sorted(names)]

web = str(export_path)
if not os.path.isdir(web):
    cands = glob.glob("runs/detect/camdetec_pest*/weights/*_web_model")
    web = cands[-1] if cands else web

with open(os.path.join(web, "labels.json"), "w") as f:
    json.dump(names, f)

print("classes (index -> name):", names)
print("web model dir:", web)
print("contents:", os.listdir(web))

## 7. Download the model bundle

In [ ]:
import shutil
shutil.make_archive("pest_model", "zip", web)
print("zipped:", os.path.abspath("pest_model.zip"))
try:
    from google.colab import files
    files.download("pest_model.zip")
except Exception:
    print("Not on Colab — grab pest_model.zip from the file browser (Kaggle: /kaggle/working/pest_model.zip).")

## 8. Install it in CamDetec

1. **Unzip** `pest_model.zip`. It contains `model.json`, `group1-shard*.bin`, and
   `labels.json`.
2. Copy **all of those files** into the repo at `public/models/pest/` so you have:
   ```
   public/models/pest/model.json
   public/models/pest/group1-shard1of*.bin
   public/models/pest/labels.json
   ```
3. Redeploy: `vercel --prod` (or commit + push).
4. In the app: **Settings → Object model → "Pest model · YOLOv8"**. It loads from
   `/models/pest/` and starts detecting your trained classes. If it fails to load,
   the app automatically falls back to COCO-SSD.

That's it — the detector seam (`lib/detector.ts` → `YoloTfjsDetector`) already knows
how to decode this model; you just supply the weights.

**Improving accuracy:** more labeled data (especially from the real camera angle),
harder negatives (empty frames), and night/IR samples help far more than extra
epochs. Retrain, re-export, replace the files in `public/models/pest/`.